# Chapter 2 (Extra Credit) — Compute MedImageInsight embeddings

Pre-compute MI2 image embeddings for the NIH 100-per-class chest X-ray subset and write them to JSONL.
Notebook 02 will train an adapter on top of these embeddings; the MCP `adapter_classify` tool will use them at inference time.

**Why precompute:** the MI2 endpoint is the slow part. Running it once and persisting the vectors lets us iterate on the adapter (and re-train it during the workshop) in seconds instead of re-hitting the endpoint every time.

**Outputs**
- `chapters/chapter2-foundation-models/extra-credit/data/embeddings/embeddings.jsonl` — one row per image (full dataset, no split).
- `chapters/chapter2-foundation-models/extra-credit/data/embeddings/embeddings.meta.json` — preprocessing kwargs, label list, embedding dim, etc. The adapter pickle is married to this recipe.

**No train/val split here** — the real eval is the agent run downstream. Notebook 02 trains on everything and uses cross-validation for a sanity number.

**Each JSONL row**
```json
{"image_path": "...", "label": "Cardiomegaly", "embedding": [0.1, -0.2, ...]}
```

## 1. Setup — wire to the chapter 2 MI2 client and the dataset

In [ ]:
import os, sys, json
from pathlib import Path

# Resolve workspace + paths from this notebook's location.
NOTEBOOK_DIR = Path.cwd()
EXTRA_CREDIT_DIR = NOTEBOOK_DIR.parent                # chapters/chapter2-foundation-models/extra-credit
CHAPTER_DIR = EXTRA_CREDIT_DIR.parent                 # chapters/chapter2-foundation-models
CHAPTERS_DIR = CHAPTER_DIR.parent                     # chapters
WORKSHOP_ROOT = CHAPTERS_DIR.parent                   # workshop repo root
REPO_ROOT = WORKSHOP_ROOT.parent                      # HealthcareAI-Examples

MI2_SERVER_DIR = WORKSHOP_ROOT / "chapters/chapter2-foundation-models/answer-key/mcp/medimageinsight-server"
DATASET_DIR = REPO_ROOT / "data/100_Datasets_Per_Class_NIH_Chest_Xray/nih_subset"
OUT_DIR = EXTRA_CREDIT_DIR / "data/embeddings"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Use chapter 2's .env so we don't fork credentials.
os.environ.setdefault("MI2_ENV_FILE", str(MI2_SERVER_DIR / ".env"))

# Import the chapter 2 MI2 client in-place (no install). We'll clean this up
# when we package things; for now, pragmatic.
sys.path.insert(0, str(MI2_SERVER_DIR))
from workshop import mi2_client  # noqa: E402

print(f"MI2 client:   {MI2_SERVER_DIR}")
print(f"Dataset:      {DATASET_DIR}")
print(f"Output dir:   {OUT_DIR}")
print(f"Endpoint set: {bool(os.environ.get('MI2_MODEL_ENDPOINT'))}")


## 2. Build the dataset frame

Folder-per-class layout under `nih_subset/`. We treat the folder name as the single label per image
(NIH images can be multi-label in the master CSV, but the 100-per-class subset is organized as single-label folders — fine for this workshop).

In [ ]:
import pandas as pd

rows = []
for cls_dir in sorted(p for p in DATASET_DIR.iterdir() if p.is_dir()):
    label = cls_dir.name
    for img in sorted(cls_dir.glob("*.png")):
        rows.append({"image_path": str(img), "label": label})

df = pd.DataFrame(rows)
print(f"Total images: {len(df)}")
print(df['label'].value_counts().sort_index())


## 3. Preprocessing recipe

Mirrors the `mi2-cxr-zero-shot-experimental` skill so the adapter sees the same pixels at train
and inference time. Recorded into the meta sidecar — the MCP `adapter_classify` tool reads these
kwargs back so the recipe can't drift.

In [ ]:
PREPROCESS = {
    "image_format": "JPEG",
    "jpeg_quality": 75,
    "resize_to": 512,
    "resample": "BICUBIC",
    "normalize_percentiles": (5, 95),
}
PREPROCESS


## 4. Embed and stream to JSONL

Threadpool over the MI2 endpoint, ordered writes. Streaming write means a partial run is recoverable —
if you stop and re-run, only missing `image_path`s are re-embedded.

In [ ]:
from joblib import Parallel, delayed
from tqdm.auto import tqdm

def _existing_paths(jsonl_path: Path) -> set[str]:
    if not jsonl_path.exists():
        return set()
    out = set()
    with jsonl_path.open() as f:
        for line in f:
            try:
                out.add(json.loads(line)["image_path"])
            except Exception:
                pass
    return out

def _embed_one(row):
    with open(row.image_path, "rb") as f:
        vec = mi2_client.get_embedding(f.read(), **PREPROCESS)
    return row, vec

def embed_all(images_df: pd.DataFrame, out_path: Path, n_jobs: int = 12) -> None:
    done = _existing_paths(out_path)
    todo_df = images_df[~images_df["image_path"].isin(done)].sample(frac=1.0, random_state=42)
    todo = list(todo_df.itertuples(index=False))
    print(f"{out_path.name}: {len(done)} already embedded, {len(todo)} to do")
    if not todo:
        return
    # generator_unordered streams results as workers finish; wrap with tqdm(total=...)
    # so the bar tracks completions, not submissions.
    results = Parallel(n_jobs=n_jobs, prefer="threads", return_as="generator_unordered")(
        delayed(_embed_one)(row) for row in todo
    )
    with out_path.open("a") as fout:
        for row, vec in tqdm(results, total=len(todo), desc=out_path.stem):
            rec = {"image_path": row.image_path, "label": row.label, "embedding": vec}
            print(json.dumps(rec), file=fout, flush=True)


In [ ]:
embed_all(df, OUT_DIR / "embeddings.jsonl")


## 5. Sanity-check + write the meta sidecar

In [ ]:
def _load(jsonl_path: Path) -> pd.DataFrame:
    with jsonl_path.open() as f:
        return pd.DataFrame(json.loads(line) for line in f)

emb = _load(OUT_DIR / "embeddings.jsonl")

embedding_dim = len(emb.iloc[0]['embedding'])
label_list = sorted(set(emb['label']))

print(f"embeddings: {len(emb)} rows, dim={embedding_dim}")
print(f"labels ({len(label_list)}): {label_list}")
print("\nclass counts:")
print(emb['label'].value_counts().sort_index())


In [ ]:
from datetime import datetime, timezone
from urllib.parse import urlparse

endpoint = os.environ.get("MI2_MODEL_ENDPOINT", "")
meta = {
    "dataset": "NIH ChestX-ray14 — 100 per class subset",
    "dataset_dir": str(DATASET_DIR),
    "n_rows": len(emb),
    "labels": label_list,
    "embedding_dim": embedding_dim,
    "preprocess": {**PREPROCESS, "normalize_percentiles": list(PREPROCESS["normalize_percentiles"])},
    "mi2_endpoint_host": urlparse(endpoint).hostname if endpoint else None,
    "created_utc": datetime.now(timezone.utc).isoformat(),
}
(OUT_DIR / "embeddings.meta.json").write_text(json.dumps(meta, indent=2))
print(json.dumps(meta, indent=2))
